# 부동산 이상 거래 탐지 — 탐색적 데이터 분석 (EDA)

**데이터**: 국토부 실거래가 + K-apt 관리비 통합 데이터셋 (`integrated_dataset_v1_with_s1.csv`)

**목적**: 중간 발표용 데이터 현황 파악 및 Autoencoder S1 결과 시각화

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/processed/integrated_dataset_v1_with_s1.csv'
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig', low_memory=False)
print(f'행: {len(df):,}  /  열: {len(df.columns)}')
df.head(3)

## 1. 데이터셋 개요

In [ ]:
print('=== 컬럼 목록 ===')
for col in df.columns:
    print(f'  {col}: {df[col].dtype}  (결측 {df[col].isna().sum():,}건)')

In [ ]:
# 기초 통계
num_cols = ['거래금액(만원)', '전용면적(㎡)', '층', '건축년도', '평당가(만원)']
df[num_cols].describe().round(1)

## 2. 거래유형 분포

In [ ]:
type_counts = df['거래유형'].value_counts()
print(type_counts.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4e8df5', '#f5a623', '#7ed321']
type_counts.plot(kind='bar', ax=ax, color=colors[:len(type_counts)], edgecolor='white')
ax.set_title('거래유형별 건수', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('거래 건수')
ax.tick_params(axis='x', rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3. 거래금액 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 분포
price = df['거래금액(만원)'].dropna()
axes[0].hist(price, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(price.median(), color='red', linestyle='--', label=f'중앙값: {price.median()/10000:.1f}억')
axes[0].set_title('거래금액 분포 (전체)', fontsize=13)
axes[0].set_xlabel('거래금액 (만원)')
axes[0].set_ylabel('거래 건수')
axes[0].legend()

# 10억 이하 구간 상세
price_u10 = price[price <= 100000]
axes[1].hist(price_u10, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(price_u10.median(), color='red', linestyle='--', label=f'중앙값: {price_u10.median()/10000:.1f}억')
axes[1].set_title('거래금액 분포 (10억 이하)', fontsize=13)
axes[1].set_xlabel('거래금액 (만원)')
axes[1].set_ylabel('거래 건수')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 계약년월 추이 (월별 거래량)

In [ ]:
monthly = df['계약년월'].astype(str).str.strip().value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly.values, marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax.set_title('월별 아파트 거래량', fontsize=13)
ax.set_xlabel('계약년월')
ax.set_ylabel('거래 건수')
# x축 라벨 간격 조정
ticks = monthly.index[::6]
ax.set_xticks(range(0, len(monthly), 6))
ax.set_xticklabels(ticks, rotation=45)
plt.tight_layout()
plt.savefig('../reports/monthly_transactions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 지역별 거래량 (상위 20 시군구)

In [ ]:
# 시군구 앞 2단어(시+구) 추출
df['_gu'] = df['시군구'].astype(str).str.split().str[:2].str.join(' ')
top_gu = df['_gu'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
top_gu.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('지역별 거래량 상위 20', fontsize=13)
ax.set_xlabel('거래 건수')
plt.tight_layout()
plt.savefig('../reports/top_regions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. K-apt 매칭 현황

In [ ]:
matched = df['단지코드'].notna().sum()
unmatched = df['단지코드'].isna().sum()
total = len(df)
rate = matched / total * 100

print(f'K-apt 매칭: {matched:,}건 ({rate:.1f}%)')
print(f'미매칭:     {unmatched:,}건 ({100-rate:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([matched, unmatched],
       labels=[f'매칭\n{matched:,}건 ({rate:.1f}%)', f'미매칭\n{unmatched:,}건 ({100-rate:.1f}%)'],
       colors=['#4e8df5', '#e0e0e0'],
       startangle=90, autopct='%1.1f%%', textprops={'fontsize': 12})
ax.set_title('K-apt 관리비 데이터 매칭률', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/kapt_match_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Autoencoder S1 결과 — 이상 거래 분포

In [ ]:
if 'S1_is_anomaly' not in df.columns:
    print('S1 컬럼 없음 — autoencoder.py 먼저 실행 필요')
else:
    anomaly_count  = df['S1_is_anomaly'].sum()
    normal_count   = len(df) - anomaly_count
    anomaly_rate   = anomaly_count / len(df) * 100

    print(f'정상 거래: {normal_count:,}건 ({100-anomaly_rate:.1f}%)')
    print(f'이상 거래: {anomaly_count:,}건 ({anomaly_rate:.1f}%)')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 파이 차트
    axes[0].pie([normal_count, anomaly_count],
                labels=[f'정상\n{normal_count:,}건', f'이상\n{anomaly_count:,}건'],
                colors=['#4e8df5', '#f45b5b'],
                startangle=90, autopct='%1.1f%%', textprops={'fontsize': 12})
    axes[0].set_title('S1 이상 거래 탐지 결과', fontsize=13)

    # 거래유형별 이상 거래 비율
    anomaly_by_type = df.groupby('거래유형')['S1_is_anomaly'].mean() * 100
    anomaly_by_type.sort_values(ascending=True).plot(
        kind='barh', ax=axes[1], color=['#4e8df5', '#f5a623', '#f45b5b'][:len(anomaly_by_type)],
        edgecolor='white'
    )
    axes[1].set_title('거래유형별 이상 거래 비율 (%)', fontsize=13)
    axes[1].set_xlabel('이상 거래 비율 (%)')
    for bar in axes[1].patches:
        axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.1f}%', va='center', fontsize=11)

    plt.tight_layout()
    plt.savefig('../reports/s1_anomaly_result.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. 이상 거래 상위 20건

In [ ]:
if 'S1_ae_error' in df.columns:
    top_anomaly = df[df['S1_is_anomaly'] == 1][
        ['시군구', '단지명', '계약년월', '거래금액(만원)', '거래유형', 'S1_ae_error']
    ].sort_values('S1_ae_error', ascending=False).head(20)

    top_anomaly['거래금액_억'] = (top_anomaly['거래금액(만원)'] / 10000).round(2)
    display_cols = ['시군구', '단지명', '계약년월', '거래금액_억', '거래유형', 'S1_ae_error']
    print('=== 재구성 오차 상위 이상 거래 20건 ===')
    print(top_anomaly[display_cols].to_string(index=False))

## 9. 재구성 오차 분포 (거래유형별)

In [ ]:
if 'S1_ae_error' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))

    for ttype, color in [('중개거래', '#4e8df5'), ('직거래', '#f5a623'), ('분양권', '#7ed321')]:
        subset = df[df['거래유형'] == ttype]['S1_ae_error'].dropna()
        if len(subset) > 0:
            # 95 퍼센타일 이하만 시각화 (극단값 제외)
            cap = subset.quantile(0.95)
            ax.hist(subset[subset <= cap], bins=80, alpha=0.6, color=color,
                    label=f'{ttype} ({len(subset):,}건)', edgecolor='white')

    ax.set_title('거래유형별 재구성 오차 분포 (95 퍼센타일 이하)', fontsize=13)
    ax.set_xlabel('재구성 오차 (MSE)')
    ax.set_ylabel('거래 건수')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('../reports/error_by_type.png', dpi=150, bbox_inches='tight')
    plt.show()

## 10. 평당가 분포 (거래유형별)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for ttype, color in [('중개거래', '#4e8df5'), ('직거래', '#f5a623'), ('분양권', '#7ed321')]:
    subset = df[df['거래유형'] == ttype]['평당가(만원)'].dropna()
    if len(subset) > 0:
        cap = subset.quantile(0.98)
        ax.hist(subset[subset <= cap], bins=80, alpha=0.6, color=color,
                label=f'{ttype}', edgecolor='white')

ax.set_title('거래유형별 평당가 분포', fontsize=13)
ax.set_xlabel('평당가 (만원/㎡)')
ax.set_ylabel('거래 건수')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../reports/price_per_area_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약 — 중간 발표 핵심 수치

In [ ]:
print('=' * 50)
print('       중간 발표 핵심 수치 요약')
print('=' * 50)
print(f'  전체 거래 건수         : {len(df):>10,} 건')
print(f'  중개거래 (정상 학습용) : {(df["거래유형"]=="중개거래").sum():>10,} 건')
print(f'  직거래                : {(df["거래유형"]=="직거래").sum():>10,} 건')
print()
print(f'  K-apt 매칭 건수        : {df["단지코드"].notna().sum():>10,} 건')
print(f'  K-apt 매칭률           : {df["단지코드"].notna().mean()*100:>9.1f} %')
print()
if 'S1_is_anomaly' in df.columns:
    anomaly = df['S1_is_anomaly'].sum()
    print(f'  S1 이상 거래 탐지      : {anomaly:>10,} 건')
    print(f'  S1 이상 거래 비율      : {anomaly/len(df)*100:>9.1f} %')
    jig_anomaly = df[(df['거래유형']=='직거래') & (df['S1_is_anomaly']==1)]
    print(f'  직거래 중 이상 건수    : {len(jig_anomaly):>10,} 건')
print('=' * 50)